# Smart Drive Creator — Integration Tests

Tests the full Smart Virtual Drive Creator pipeline:
- `POST /api/ai/vision/llama-scout` — Groq Llama 4 Scout vision endpoint on Server
- `smart_drive_scan` agent tool — NTFS MFT scan + AI file analysis
- `smart_drive_build` agent tool — virtual drive creation with approval flow
- Full agent flow: natural-language request → scan → review → drive created
- Hierarchical drives: multiple sub-folders inside one drive

---
**Pre-conditions**
```bash
# 1. Docker server (from Server/):
docker compose -f docker/docker-compose.dev.yml up --build -d

# 2. APP Flask (from APP/src/):
python main.py
```
**Dependencies:**
```bash
pip install httpx requests pillow
```

---
| # | What is tested |
|---|----------------|
| 1 | Vision endpoint: POST image → Groq Llama Scout → description + tags |
| 2 | `smart_drive_scan` tool: scan temp folder, verify AI descriptions |
| 3 | `smart_drive_build` tool: auto-approve, verify drive on disk |
| 4 | Full agent flow: natural-language → scan → build |
| 5 | Hierarchical drive: sub-folders inside the drive |

In [23]:
# ── 0. Configuration ────────────────────────────────────────────────────────────

SERVER_URL       = 'http://localhost:8000'   # FastAPI — Docker or local
APP_URL          = 'http://localhost:5000'   # Flask   — APP local
TOOL_CALLBACK_URL = 'http://host.docker.internal:5000/api/tools/execute'

TEST_EMAIL    = 'smart_drive_test@example.com'
TEST_PASSWORD = 'TestPass123!'

print(f'Server: {SERVER_URL}')
print(f'APP:    {APP_URL}')

Server: http://localhost:8000
APP:    http://localhost:5000


In [24]:
# ── 1. Service health check ──────────────────────────────────────────────────────

import httpx

async def check_services():
    errors = []
    async with httpx.AsyncClient(timeout=5) as c:
        try:
            r = await c.get(f'{SERVER_URL}/health')
            print(f'OK  Docker server -> {r.json()}')
        except Exception as e:
            errors.append('Docker server')
            print(f'ERR Docker server: {e}')

        try:
            r = await c.get(f'{APP_URL}/api/health')
            print(f'OK  APP Flask -> {r.json()}')
        except Exception as e:
            errors.append('APP Flask')
            print(f'ERR APP Flask: {e}')

    if errors:
        print(f'\nSTOP: {errors} offline')
        return False
    print('\nAll services online.')
    return True

SERVICES_OK = await check_services()

ERR Docker server: Expecting value: line 1 column 1 (char 0)
OK  APP Flask -> {'message': 'Python backend is running', 'status': 'ok'}

STOP: ['Docker server'] offline


In [25]:
# ── 2. Create test fixtures ──────────────────────────────────────────────────────
# One temp folder with: 2 PNG images, 1 JPEG, 1 WAV audio file, 1 PDF

import tempfile, struct, zlib, wave, os
from pathlib import Path
from PIL import Image

TEST_TEMP_DIR = Path(tempfile.mkdtemp(prefix='smart_drive_test_'))
print(f'Test dir: {TEST_TEMP_DIR}')

# 2 PNG images with distinct colors (simulate different content)
img_dog = TEST_TEMP_DIR / 'dog_portrait.png'
img_cat = TEST_TEMP_DIR / 'cat_photo.png'
img_jpg = TEST_TEMP_DIR / 'landscape.jpg'

Image.new('RGB', (200, 200), color=(139, 90, 43)).save(str(img_dog), 'PNG')    # brownish
Image.new('RGB', (200, 200), color=(180, 180, 200)).save(str(img_cat), 'PNG')  # greyish
Image.new('RGB', (200, 200), color=(34, 139, 34)).save(str(img_jpg), 'JPEG')   # green

# 1 WAV audio (1 second of silence)
audio_file = TEST_TEMP_DIR / 'sample_speech.wav'
with wave.open(str(audio_file), 'w') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(16000)
    wf.writeframes(b'\x00' * 16000 * 2)  # 1 sec silence

# 1 TXT document
doc_file = TEST_TEMP_DIR / 'business_report.txt'
doc_file.write_text('Q1 Revenue Report\n\nSales increased by 15% in Q1 2025.\nKey clients: Acme Corp, Beta Ltd.', encoding='utf-8')

# Virtual drive output folder
DRIVES_OUTPUT = TEST_TEMP_DIR / 'virtual_drives'
DRIVES_OUTPUT.mkdir()

print(f'Created: {[f.name for f in TEST_TEMP_DIR.iterdir()]}')

Test dir: C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv
Created: ['business_report.txt', 'cat_photo.png', 'dog_portrait.png', 'landscape.jpg', 'sample_speech.wav', 'virtual_drives']


In [26]:
# ── 3. Auth + agent key ──────────────────────────────────────────────────────────

import httpx

async def auth() -> str:
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        r = await c.post('/api/auth/register', json={'email': TEST_EMAIL, 'password': TEST_PASSWORD})
        if r.status_code == 201:
            print(f'New user: {TEST_EMAIL}')
            return r.json()['access_token']
        r = await c.post('/api/auth/login', data={'username': TEST_EMAIL, 'password': TEST_PASSWORD})
        r.raise_for_status()
        print(f'Login: {TEST_EMAIL}')
        return r.json()['access_token']

async def get_agent_key(jwt: str) -> str:
    headers = {'Authorization': f'Bearer {jwt}'}
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        r = await c.post('/api/agent/register', headers=headers)
        if r.status_code == 201:
            return r.json()['api_key']
        r = await c.get('/api/agent/key', headers=headers)
        r.raise_for_status()
        return r.json()['api_key']

JWT       = await auth()
AGENT_KEY = await get_agent_key(JWT)
AGENT_HEADERS = {'X-API-Key': AGENT_KEY}
print(f'Agent key: {AGENT_KEY[:8]}...')

Login: smart_drive_test@example.com
Agent key: 7b082724...


In [27]:
# ── 4. Fetch real tools from APP + inject callback URL ────────────────────────────

import httpx

async def fetch_tools() -> list:
    async with httpx.AsyncClient(base_url=APP_URL, timeout=10) as c:
        r = await c.get('/api/tools')
        r.raise_for_status()
        tools = r.json()
    for t in tools:
        t['callback_url'] = TOOL_CALLBACK_URL
    return tools

TOOLS = await fetch_tools()
tool_names = [t['name'] for t in TOOLS]
print(f'{len(TOOLS)} tools loaded: {tool_names}')

assert 'smart_drive_scan'  in tool_names, 'smart_drive_scan not found in APP tools'
assert 'smart_drive_build' in tool_names, 'smart_drive_build not found in APP tools'
print('\nOK: both smart drive tools are registered.')

17 tools loaded: ['ask_user', 'hello', 'image_converter', 'remove_background', 'image_to_svg', 'video_converter', 'video_compressor', 'audio_converter', 'drive_creator', 'space_analyzer', 'pdf_merger', 'model_converter', 'document_converter', 'subtitle_generator', 'document_analytics', 'smart_drive_scan', 'smart_drive_build']

OK: both smart drive tools are registered.


In [28]:
# ── 4b. Subset for smart-drive agent tests ───────────────────────────────────────
# Tests 4 & 5 only need 3 tools. Sending all 17 tool descriptions to the
# planning agent exceeds the Groq free-tier 8000 TPM limit (8564/8585 tokens).
SD_TOOLS = [t for t in TOOLS if t['name'] in ('ask_user', 'smart_drive_scan', 'smart_drive_build')]
print(f'SD_TOOLS ({len(SD_TOOLS)}): {[t["name"] for t in SD_TOOLS]}')

SD_TOOLS (3): ['ask_user', 'smart_drive_scan', 'smart_drive_build']


In [29]:
# ── 5. Auto-approver (approves smart_drive_build with full file list) ─────────────

import threading, time, requests as _req

_approval_log = []
_stop_approver = threading.Event()

def _approver_loop():
    while not _stop_approver.is_set():
        try:
            r = _req.get(f'{APP_URL}/api/tools/pending', timeout=3)
            if r.ok:
                for item in r.json():
                    req_id = item['id']
                    inp    = dict(item['input'])
                    tool   = item['tool']

                    if tool == 'ask_user':
                        input_type = inp.get('input_type', 'text')
                        if input_type == 'folder':
                            inp['answer'] = str(DRIVES_OUTPUT)
                        elif input_type == 'yesno':
                            inp['answer'] = 'yes'
                        else:
                            inp['answer'] = 'Smart Drive Test'
                    elif tool == 'smart_drive_build':
                        # Ensure outputPath is set to our test drive folder
                        inp['outputPath'] = str(DRIVES_OUTPUT)

                    ar = _req.post(f'{APP_URL}/api/tools/approve/{req_id}', json={'input': inp}, timeout=3)
                    if ar.ok:
                        _approval_log.append({'id': req_id, 'tool': tool})
                        print(f'[auto-approver] {tool} approved (req {req_id[:8]}...)')
        except Exception:
            pass
        time.sleep(0.4)

_approver_thread = threading.Thread(target=_approver_loop, daemon=True)
_approver_thread.start()
print('Auto-approver started.')

Auto-approver started.


In [30]:
# ── 6. Streaming helpers (same as test_planning_agent.ipynb) ─────────────────────

import json, httpx

async def create_chat(title='', tools=None) -> str:
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        r = await c.post('/api/agent/chats', headers=AGENT_HEADERS,
                         json={'title': title, 'tools': tools or TOOLS})
        r.raise_for_status()
        return r.json()['chat_id']

async def stream_message(chat_id: str, message: str, tools=None):
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=300) as c:
        async with c.stream('POST', f'/api/agent/chats/{chat_id}/message',
                            headers={**AGENT_HEADERS, 'Accept': 'text/event-stream'},
                            json={'message': message, 'tools': tools if tools is not None else TOOLS}) as resp:
            resp.raise_for_status()
            async for line in resp.aiter_lines():
                if not line.startswith('data: '): continue
                payload = line[6:]
                if payload.strip() == '[DONE]': return
                try:
                    yield json.loads(payload)
                except json.JSONDecodeError:
                    pass

_streaming = False
def render(evt):
    global _streaming
    t = evt.get('type', '?')
    if t in ('llm_chunk', 'final_chunk'):
        if not _streaming: print(); _streaming = True
        print(evt.get('content', ''), end='', flush=True)
        return
    if _streaming: print(); _streaming = False
    if t == 'plan':
        steps = evt.get('steps', [])
        print(f'\n[PLAN] {len(steps)} steps')
        for s in steps:
            icon = '[TOOL]' if s.get('type') == 'tool' else '[LLM] '
            lbl  = s.get('tool', 'LLM') if s.get('type') == 'tool' else 'LLM'
            print(f'   {s["id"]}. {icon} {lbl}: {s["description"]}')
        return
    if t == 'final':
        print(f'\n{"="*55}\nFINAL:\n{evt.get("response","")}\n{"="*55}')
        return
    ICON = {'status':'[...]','tool_call':'[>>] ','tool_result':'[<<] ','tool_error':'[ERR]','step_start':'[STEP]','step_done':'  [v]','error':'[!!!]'}
    icon = ICON.get(t, ' [-] ')
    msg  = evt.get('message','')
    if msg: print(f'   {icon} {msg}')

async def run(chat_id, message, tools=None):
    events, plan, tool_calls, final = [], [], [], ''
    async for evt in stream_message(chat_id, message, tools):
        events.append(evt)
        render(evt)
        if evt.get('type') == 'plan':      plan = evt.get('steps', [])
        if evt.get('type') == 'tool_call': tool_calls.append(evt.get('tool'))
        if evt.get('type') == 'final':     final = evt.get('response', '')
    return {'events': events, 'plan': plan, 'tools_called': tool_calls, 'final': final}

print('Helpers OK')

Helpers OK


---
## TEST 1 — Vision endpoint: POST image → Groq Llama Scout
**Expected**: `description` is a non-empty string, `tags` is a list.

In [31]:
import requests as _req

with open(img_dog, 'rb') as fh:
    img_bytes = fh.read()

r = _req.post(
    f'{SERVER_URL}/api/ai/vision/llama-scout',
    files={'file': ('dog_portrait.png', img_bytes, 'image/png')},
    data={'max_tokens': '150'},
    timeout=(10, 60),
)
print(f'Status: {r.status_code}')
if r.ok:
    data = r.json()
    print(f'Model:       {data.get("model")}')
    print(f'Latency:     {data.get("latency_s")} s')
    print(f'Description: {data.get("description", "")[:200]}')
    print(f'Tags:        {data.get("tags", [])[:8]}')

    ok1 = bool(data.get('description')) and isinstance(data.get('tags'), list)
else:
    print(f'Error: {r.text[:300]}')
    ok1 = False

status1 = 'PASS' if ok1 else 'FAIL'
print(f'\nTEST 1 Result: {status1}')

Status: 200
Model:       meta-llama/llama-4-scout-17b-16e-instruct
Latency:     1.382 s
Description: The image presents a solid, uniform brown color that fills the entire frame. The absence of any discernible objects or features creates a sense of simplicity and minimalism.

**Key Features:**

* **Co
Tags:        ['image', 'presents', 'solid', 'uniform', 'brown', 'color', 'that', 'fills']

TEST 1 Result: PASS


---
## TEST 2 — `smart_drive_scan` tool via direct APP call
**Expected**: tool returns files list with at least 1 analyzed image with `ai_description`.

In [32]:
import requests as _req, json

r = _req.post(
    f'{APP_URL}/api/tools/execute',
    json={
        'tool': 'smart_drive_scan',
        'input': {
            'sourceFolder': str(TEST_TEMP_DIR),
            'extensions': ['.png', '.jpg', '.wav', '.txt'],
            'maxAnalyze': 10,
        },
    },
    timeout=(10, 180),
)
print(f'Status: {r.status_code}')
raw = r.json()
result = json.loads(raw.get('result', '{}'))

print(f'Success:     {result.get("success")}')
print(f'Total found: {result.get("total_found")}')
print(f'Analyzed:    {result.get("analyzed")}')
print(f'Scan method: {result.get("scan_method")}')

files = result.get('files', [])
for f in files:
    desc = (f.get('ai_description') or '')[:80]
    err  = f.get('ai_error')
    print(f'  [{f["type"]:8s}] {f["filename"]:30s} desc={desc!r:.60}  err={err}')

images_with_desc = [f for f in files if f.get('type') == 'image' and f.get('ai_description')]
ok2 = result.get('success') and len(files) > 0
status2 = 'PASS' if ok2 else 'FAIL'
print(f'\nTEST 2 Result: {status2}')
if not images_with_desc:
    print('  NOTE: No images with AI descriptions — check that Server is running (Groq API key set?)')

Status: 200
Success:     True
Total found: 5
Analyzed:    5
Scan method: walk
  [document] business_report.txt            desc='Q1 Revenue Report\n\nSales increased by 15% in Q1 2025.\nKe  err=None
  [image   ] cat_photo.png                  desc='The image presents a solid, uniform light purple color, dev  err=None
  [image   ] dog_portrait.png               desc='The image presents a solid, uniform brown color that fills   err=None
  [image   ] landscape.jpg                  desc='The image presents a solid green background, devoid of any   err=None
  [audio   ] sample_speech.wav              desc='Thank you.'  err=None

TEST 2 Result: PASS


---
## TEST 3 — `smart_drive_build` via agent (auto-approved → drive created on disk)
**Expected**: agent calls `smart_drive_build`, auto-approver confirms, drive folder appears on disk.

In [33]:
_approval_log.clear()

# Use direct tool execute so we can control the exact input
import requests as _req, json

build_input = {
    'driveName':  'Test Image Drive',
    'outputPath': str(DRIVES_OUTPUT),
    'action':     'shortcuts',
    'files': [
        {'path': str(img_dog), 'folder': '', 'ai_description': 'brownish color image'},
        {'path': str(img_cat), 'folder': '', 'ai_description': 'greyish color image'},
    ],
}

import threading, time
_build_done = threading.Event()
_build_result = {}

def _call_build():
    r = _req.post(f'{APP_URL}/api/tools/execute',
                  json={'tool': 'smart_drive_build', 'input': build_input},
                  timeout=130)
    _build_result.update(r.json())
    _build_done.set()

t = threading.Thread(target=_call_build, daemon=True)
t.start()
print('Calling smart_drive_build (waiting for auto-approver)...')

_build_done.wait(timeout=30)
raw3 = json.loads(_build_result.get('result', '{}'))
print(f'Result: {json.dumps(raw3, indent=2)}')

drive_path = raw3.get('drivePath', '')
drive_exists = bool(drive_path) and os.path.isdir(drive_path)

print(f'\nDrive path:   {drive_path}')
print(f'Drive exists: {drive_exists}')
if drive_exists:
    contents = list(os.scandir(drive_path))
    print(f'Drive contents ({len(contents)} items):')
    for e in contents:
        print(f'  {e.name}')

ok3 = raw3.get('success') and drive_exists
status3 = 'PASS' if ok3 else 'FAIL'
print(f'\nTEST 3 Result: {status3}')

Calling smart_drive_build (waiting for auto-approver)...
[auto-approver] smart_drive_build approved (req 921a78f2...)
Result: {
  "success": true,
  "drivePath": "C:\\Users\\redis\\AppData\\Local\\Temp\\smart_drive_test_r3y7ghdv\\virtual_drives\\Test Image Drive",
  "total": 2,
  "succeeded": 2,
  "failed": 0,
  "folders_created": [],
  "results": [
    {
      "path": "C:\\Users\\redis\\AppData\\Local\\Temp\\smart_drive_test_r3y7ghdv\\dog_portrait.png",
      "folder": null,
      "success": true
    },
    {
      "path": "C:\\Users\\redis\\AppData\\Local\\Temp\\smart_drive_test_r3y7ghdv\\cat_photo.png",
      "folder": null,
      "success": true
    }
  ]
}

Drive path:   C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv\virtual_drives\Test Image Drive
Drive exists: True
Drive contents (3 items):
  .drive_config.json
  cat_photo.png.lnk
  dog_portrait.png.lnk

TEST 3 Result: PASS


---
## TEST 4 — Full agent flow: natural-language → scan → build
**Expected**: agent calls `smart_drive_scan` then `smart_drive_build`; drive created on disk.

In [34]:
import asyncio
_approval_log.clear()

chat4 = await create_chat(title='Test 4 - Full agent flow', tools=SD_TOOLS)
msg4  = (
    f'I want a virtual drive with all the PNG images from the folder "{str(TEST_TEMP_DIR)}". '
    f'Save the drive in "{str(DRIVES_OUTPUT)}". Call it "My PNG Drive" and use shortcuts.'
)

print(f'Chat: {chat4}')
print(f'User: {msg4}')
print('=' * 60)

r4 = await run(chat4, msg4, tools=SD_TOOLS)

scan_called  = 'smart_drive_scan'  in r4['tools_called']
build_called = 'smart_drive_build' in r4['tools_called']
n_approvals4 = len(_approval_log)

# Check a drive was created
drives_made = [d for d in DRIVES_OUTPUT.iterdir() if d.is_dir()] if DRIVES_OUTPUT.exists() else []

print(f'\nVERIFICATION TEST 4')
print(f'  Tools called:         {r4["tools_called"]}')
print(f'  smart_drive_scan:     {scan_called}   (expected: True)')
print(f'  smart_drive_build:    {build_called}  (expected: True)')
print(f'  Auto-approvals:       {n_approvals4}')
print(f'  Drive folders found:  {[d.name for d in drives_made]}')

ok4 = scan_called and build_called
status4 = 'PASS' if ok4 else 'FAIL'
print(f'\n  Result: {status4}')

Chat: d2dcdf48-645c-4e7c-8d74-7fef52a263fa
User: I want a virtual drive with all the PNG images from the folder "C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv". Save the drive in "C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv\virtual_drives". Call it "My PNG Drive" and use shortcuts.
   [...] Analizez cererea și creez planul de execuție…

[PLAN] 2 steps
   1. [TOOL] smart_drive_scan: Scan the source folder for all PNG images and let the AI analyze them.
   2. [TOOL] smart_drive_build: Create a virtual drive named "My PNG Drive" containing the PNG files found, using shortcuts and storing the drive in the specified output folder.
   [STEP] Pasul 1/2: Scan the source folder for all PNG images and let the AI analyze them.
   [>>]  Apelez tool: smart_drive_scan(sourceFolder="C:\\Users\\redis\\AppData\\Local\\Temp\\smart_drive_test_r3y7ghdv"…)
   [<<]  Tool smart_drive_scan a returnat un rezultat
     [v] Pasul 1 finalizat
   [STEP] Pasul 2/2: Create a virtual dri

---
## TEST 5 — Hierarchical drive: agent creates sub-folders inside the drive
**Expected**: agent proposes a hierarchy (images folder + docs folder); `smart_drive_build` receives `files` with `folder` values; sub-directories created inside the drive.

In [35]:
import asyncio
_approval_log.clear()

chat5 = await create_chat(title='Test 5 - Hierarchical drive', tools=SD_TOOLS)
msg5  = (
    f'I want a virtual drive called "Organized Drive" in "{str(DRIVES_OUTPUT)}" with 2 sub-folders: '
    f'one called "Images" containing all PNG and JPEG files from "{str(TEST_TEMP_DIR)}", '
    f'and one called "Documents" containing all TXT files from the same folder. '
    f'Use shortcuts.'
)

print(f'Chat: {chat5}')
print(f'User: {msg5}')
print('=' * 60)

r5 = await run(chat5, msg5, tools=SD_TOOLS)

scan_called5  = 'smart_drive_scan'  in r5['tools_called']
build_called5 = 'smart_drive_build' in r5['tools_called']

# Look for sub-folders in any newly created drive
all_drives = [d for d in DRIVES_OUTPUT.iterdir() if d.is_dir()] if DRIVES_OUTPUT.exists() else []
any_has_subfolders = any(
    any(e.is_dir() for e in d.iterdir() if e.name != '.drive_config.json')
    for d in all_drives
)

print(f'\nVERIFICATION TEST 5')
print(f'  Tools called:      {r5["tools_called"]}')
print(f'  scan called:       {scan_called5}')
print(f'  build called:      {build_called5}')
print(f'  Drive folders:     {[d.name for d in all_drives]}')
for d in all_drives:
    items = list(d.iterdir())
    print(f'    {d.name}/: {[e.name for e in items]}')

ok5 = scan_called5 and build_called5
status5 = 'PASS' if ok5 else 'FAIL'
print(f'\n  Result: {status5}')
if not any_has_subfolders:
    print('  NOTE: No sub-folders detected — agent may have built a flat drive. Check the final response.')

Chat: 0c115857-8fbc-4cb7-9199-8e04bbbd9863
User: I want a virtual drive called "Organized Drive" in "C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv\virtual_drives" with 2 sub-folders: one called "Images" containing all PNG and JPEG files from "C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv", and one called "Documents" containing all TXT files from the same folder. Use shortcuts.
   [...] Analizez cererea și creez planul de execuție…

[PLAN] 3 steps
   1. [TOOL] smart_drive_scan: Scan the source folder for image (PNG, JPG, JPEG) and text files
   2. [LLM]  LLM: Create the file list for the virtual drive, assigning each file to the proper sub‑folder
   3. [TOOL] smart_drive_build: Build the virtual drive with shortcuts, placing files into the requested sub‑folders
   [STEP] Pasul 1/3: Scan the source folder for image (PNG, JPG, JPEG) and text files
   [>>]  Apelez tool: smart_drive_scan(sourceFolder="C:\\Users\\redis\\AppData\\Local\\Temp\\smart_drive_test_r3y7g

---
## Final Summary

In [36]:
print('=' * 62)
print('  SUMAR TESTE SMART DRIVE CREATOR')
print('=' * 62)
print(f'  Test 1 — Vision endpoint (Groq Llama Scout)  {status1}')
print(f'  Test 2 — smart_drive_scan direct execute     {status2}')
print(f'  Test 3 — smart_drive_build (auto-approved)   {status3}')
print(f'  Test 4 — Full agent flow (scan + build)      {status4}')
print(f'  Test 5 — Hierarchical drive (sub-folders)    {status5}')
print('=' * 62)

all_pass = all(s == 'PASS' for s in [status1, status2, status3, status4, status5])
print(f'\n{"TOATE TESTELE AU TRECUT" if all_pass else "UNELE TESTE AU ESUAT — verifica output-ul de mai sus"}')

  SUMAR TESTE SMART DRIVE CREATOR
  Test 1 — Vision endpoint (Groq Llama Scout)  PASS
  Test 2 — smart_drive_scan direct execute     PASS
  Test 3 — smart_drive_build (auto-approved)   PASS
  Test 4 — Full agent flow (scan + build)      PASS
  Test 5 — Hierarchical drive (sub-folders)    PASS

TOATE TESTELE AU TRECUT


In [37]:
# ── Cleanup ────────────────────────────────────────────────────────────────────

_stop_approver.set()
print('Auto-approver stopped.')

import shutil
shutil.rmtree(TEST_TEMP_DIR, ignore_errors=True)
print(f'Temp files deleted: {TEST_TEMP_DIR}')

Auto-approver stopped.
Temp files deleted: C:\Users\redis\AppData\Local\Temp\smart_drive_test_r3y7ghdv
